In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/weather/search"

payload = {
    "query": "severe thunderstorms and damaging winds",
    "top_k": 5
}

response = requests.post(
    url,
    json=payload,
    headers={
        "Authorization": f"Bearer {app_token}",
        "Content-Type": "application/json",
    },
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/weather/search"

queries = [
    "severe thunderstorms and damaging winds",
    "risk of flooding near rivers",
    "clear sunny weather"
]


for query in queries:
    payload = {
        "query": query,
        "top_k": 5
    }

    response = requests.post(
        url,
        json=payload,
        headers={
            "Authorization": f"Bearer {app_token}",
            "Content-Type": "application/json",
        },
    )

    print("\nQUERY:", query)
    print("STATUS:", response.status_code)
    print(json.dumps(response.json(), indent=2))

In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/weather/search"

tests = [
    {
        "name": "No filter",
        "payload": {
            "query": "severe thunderstorms and damaging winds",
            "top_k": 5
        }
    },
    {
        "name": "Alerts only",
        "payload": {
            "query": "severe thunderstorms and damaging winds",
            "top_k": 5,
            "source_type": "alert"
        }
    },
    {
        "name": "Forecasts only",
        "payload": {
            "query": "severe thunderstorms and damaging winds",
            "top_k": 5,
            "source_type": "forecast"
        }
    }
]

for test in tests:
    response = requests.post(
        url,
        json=test["payload"],
        headers={
            "Authorization": f"Bearer {app_token}",
            "Content-Type": "application/json",
        },
    )

    print("\n" + "=" * 60)
    print(test["name"])
    print("Status:", response.status_code)
    print(json.dumps(response.json(), indent=2))